# Chain of Thought (CoT) — baseline conversacional

**Idea:** recomendador conversacional que **razona paso a paso** sobre los gustos y
restricciones del usuario antes de producir la recomendación. Es un baseline de
*razonamiento* puro (sin memoria, sin warmup, sin tools), a diferencia de Reflexion
y Dynamic Cheatsheet que acumulan memoria.

**Consistencia con el resto del proyecto:**
- Datos y métricas se importan desde `utils` (archivo central acordado en la sesión).
- Mismo `build_context` (conversación completa truncada antes del último turno del recomendador → sin fuga del GT).
- Mismas métricas con **soft-matching**: Recall@1, Recall@5, MRR, Novelty, BERTScore (threshold=0.90, gt_field=`gt_accepted`).
- Mismo sampling: `sample_conversations(test, n=300, seed=42)`.

**Regla crítica de prompt (sesión 25-06):** bajo **ninguna circunstancia** recomendar
una película ya mencionada en la conversación.

## 0. Dependencias

In [1]:
# === Dependencias (correr una vez) ===
!pip install -q transformers accelerate bitsandbytes sentence-transformers rapidfuzz evaluate bert_score
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: Tesla T4


## 1. Montar Drive y ubicarse en la carpeta del proyecto

In [2]:
# === Drive: montar y posicionarse en la carpeta del proyecto ===============
# La carpeta esta en "Compartidos conmigo", que Colab NO monta directamente.
# Pasos (una sola vez, en la web de Drive):
#   click derecho sobre "Proyecto RecSys" -> Organizar -> Anadir acceso directo
#   -> elegir "Mi unidad". Asi aparece bajo MyDrive (es solo un puntero).
import os, sys
from google.colab import drive
drive.mount("/content/drive")

# Carpeta compartida via acceso directo en Mi unidad:
PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys"
assert os.path.isdir(PROJECT_DIR), (
    "No existe " + PROJECT_DIR + ". Crea el acceso directo en Mi unidad "
    "(click derecho en la carpeta de Compartidos conmigo -> Anadir acceso directo).")
os.chdir(PROJECT_DIR)            # ahora utils.ipynb/utils.py se ven en el cwd
sys.path.insert(0, PROJECT_DIR)  # respaldo para el import

PATH = PROJECT_DIR + "/"         # carpeta donde guardamos resultados
print("Trabajando en:", os.getcwd())
print("Archivos:", [f for f in os.listdir() if f.startswith("utils")])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Trabajando en: /content/drive/.shortcut-targets-by-id/1DVeLSUjM_ZgsdoEI3aeO7kyHJTdoFuLR/Proyecto RecSys
Archivos: ['utils.ipynb', 'utils.py']


## 2. utils + datos (sin fuga)

In [3]:
# === Bootstrap de utils ====================================================
# Estamos parados en PROJECT_DIR (celda anterior), donde vive utils.ipynb.
# Si no existe utils.py lo generamos desde el .ipynb y luego importamos.
import os, json

# if not os.path.exists("utils.py"):
#     if os.path.exists("utils.ipynb"):
#         nb = json.load(open("utils.ipynb", encoding="utf-8"))
#         with open("utils.py", "w", encoding="utf-8") as f:
#             for c in nb["cells"]:
#                 if c["cell_type"] == "code":
#                     f.write("".join(c["source"]) + "\n\n")
#         print("utils.py generado desde utils.ipynb")
#     else:
#         raise FileNotFoundError(
#             "No encuentro utils.ipynb ni utils.py en " + os.getcwd() +
#             ". Revisa PROJECT_DIR en la celda anterior.")

from utils import (
    download_dataset, load_parsed, sample_conversations,
    build_context, build_global_catalog,
    build_train_freq, build_recommender_references,
    evaluate_soft, evaluate_novelty, evaluate_bertscore, run_full_evaluation,
)

# --- Descarga y carga ---
DATASET = "pearl"            # "pearl" o "redial"
paths = download_dataset(DATASET, splits=("train", "test"))
train_parsed = load_parsed(paths["train"], dataset=DATASET)
test_parsed  = load_parsed(paths["test"], dataset=DATASET)
print(f"Train: {len(train_parsed)} | Test: {len(test_parsed)}")

# --- Sampling reproducible (mismo seed para todos los metodos) ---
N_EVAL = 300
GT_FIELD = "gt_accepted"
eval_sample = sample_conversations(test_parsed, n=N_EVAL, seed=42, gt_field=GT_FIELD)
print(f"Eval sample: {len(eval_sample)} conversaciones")

pearl_train.json ya existe, omitiendo descarga.
pearl_test.json ya existe, omitiendo descarga.
Train: 50000 | Test: 2277
Sampled 300 conversaciones con gt_accepted no vacío (seed=42)
Eval sample: 300 conversaciones


## 3. Modelo base — Qwen3-8B 4-bit

In [4]:
# === Modelo base: Qwen3-8B 4-bit (cabe en T4) ==============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, json, re

MODEL_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto")

def call_llm(messages, max_tokens=512):
    """Inferencia determinista (greedy). enable_thinking=False: el razonamiento
    CoT lo elicitamos explicitamente en el prompt, no via el bloque <think>."""
    text = tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

def parse_last_json(raw):
    """Extrae el ULTIMO objeto JSON plano de la salida. Robusto a CoT: el
    razonamiento en texto plano va antes del JSON final y se ignora.
    Asume que el objeto de recomendacion no tiene llaves anidadas."""
    matches = re.findall(r"\{[^{}]*\}", raw, re.DOTALL)
    for m in reversed(matches):
        try:
            return json.loads(m)
        except Exception:
            continue
    return {}

def dedup_titles(titles):
    """Quita titulos duplicados preservando orden (normaliza sin anio)."""
    seen, out = set(), []
    for t in titles:
        k = t.split(" (")[0].strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(t)
    return out


print("Modelo cargado.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado.


## 4. Artefactos de métricas

In [5]:
# === Modelo de embeddings + artefactos para Novelty/BERTScore ==============
from sentence_transformers import SentenceTransformer

device_embed = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)

train_freq, n_train = build_train_freq(paths["train"], dataset=DATASET)     # Novelty
human_refs = build_recommender_references(paths["test"], dataset=DATASET)    # BERTScore (alineado a test_parsed)

# Parametros fijos (identicos en todos los notebooks)
THRESHOLD = 0.90

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Train freq construido: 9303 títulos únicos en 50000 diálogos
Referencias construidas: 2277 diálogos


## 5. Componente CoT

In [6]:
# === Recomendador Chain of Thought =========================================
# El modelo PRIMERO razona en texto plano (gustos, generos, restricciones y
# que titulos ya fueron mencionados y deben excluirse) y LUEGO emite un unico
# JSON con las 5 recomendaciones. parse_last_json toma ese JSON final.
# Formato de "message" acordado por el grupo: explica la 1a recomendacion con un
# motivo y luego menciona las otras 4 (consistente con ACE para BERTScore).

COT_SYSTEM = (
    "You are a conversational movie recommender that reasons step by step.\n\n"
    "CRITICAL RULE: Under NO circumstances may you recommend a movie that "
    "has already been mentioned by ANYONE (User or Assistant) anywhere in "
    "the conversation history. Every one of your 5 recommendations must be "
    "a movie NOT YET discussed in this conversation — genuinely new "
    "suggestions the user hasn't heard yet.\n\n"
    "Follow exactly these steps in PLAIN TEXT (no JSON yet):\n"
    "1. List the movie titles already mentioned by anyone (these are FORBIDDEN as outputs).\n"
    "2. Infer the user's tastes: preferred genres, tone/mood, era, and any explicit constraints.\n"
    "3. Pick 5 NEW movies that best match, none of them in the forbidden list, best match first.\n\n"
    "After the reasoning, output EXACTLY ONE JSON object on its own line and nothing after it:\n"
    '{"message": "I recommend [Title (Year)] because [brief reason]. '
    'You might also enjoy [T2], [T3], [T4], and [T5].", '
    '"structured_recommendations": ["Title (Year)", "T2", "T3", "T4", "T5"]}\n'
    "Each recommendation must be formatted as 'Title (Year)'. Exactly 5, best match first."
)

def cot_recommend(dialogue, max_tokens=768):
    messages = [
        {"role": "system", "content": COT_SYSTEM},
        {"role": "user",   "content": build_context(dialogue)},
    ]
    raw = call_llm(messages, max_tokens=max_tokens)
    data = parse_last_json(raw)
    recs = dedup_titles(data.get("structured_recommendations", []) or [])
    return recs, data.get("message", ""), raw

# --- Smoke test en 1 ejemplo ---
_recs, _msg, _raw = cot_recommend(eval_sample[0])
print("Recs:", _recs)
print("Msg :", _msg)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Recs: ['The Fugitive (1993)', 'The Silence of the Lambs (1991)', 'The Departed (2006)', 'Prisoners (2013)', 'The Prestige (2006)']
Msg : I recommend 'The Fugitive' (1993) because it is a thrilling and unpredictable plot with strong performances by Harrison Ford. You might also enjoy 'The Silence of the Lambs' (1991), 'The Departed' (2006), 'Prisoners' (2013), and 'The Prestige' (2006).


## 6. Evaluación

In [7]:
# === Evaluacion en test (idx = original_idx para alinear con test_parsed) ===
# IMPORTANTE: guardamos "idx" = original_idx del dialogo en test_parsed, de modo
# que parsed[o["idx"]] (GT) y human_refs[o["idx"]] (BERTScore) queden alineados.
import time

def cot_eval(sample, out_path, save_every=25):
    outs, done = [], set()
    if os.path.exists(out_path):
        outs = json.load(open(out_path))
        done = {o["sample_pos"] for o in outs}
    t0 = time.time()
    for pos, d in enumerate(sample):
        if pos in done:
            continue
        recs, msg, raw = cot_recommend(d)
        outs.append({
            "sample_pos": pos,
            "idx": d["original_idx"],
            "predicted": recs,
            "message": msg,
            "raw": raw,
        })
        n = len(outs)
        if n % save_every == 0:
            json.dump(outs, open(out_path, "w"))
            eta = ((time.time()-t0)/max(n-len(done),1))*(len(sample)-n)/60
            print(f"{n}/{len(sample)} | ETA {eta:.1f} min")
    json.dump(outs, open(out_path, "w"))
    return outs

out_cot = cot_eval(eval_sample, PATH + f"cot_results_{DATASET}.json")

print("\n=== Chain of Thought — metricas (soft-matching) ===")
metrics_cot = run_full_evaluation(
    out_cot, test_parsed, EMBED_MODEL,
    train_freq, n_train, human_refs,
    threshold=THRESHOLD, gt_field=GT_FIELD, k_list=(1, 5),
)
json.dump(metrics_cot, open(PATH + f"cot_metrics_{DATASET}.json", "w"), indent=2)
print(metrics_cot)

25/300 | ETA 197.3 min
50/300 | ETA 182.4 min
75/300 | ETA 160.9 min
100/300 | ETA 140.4 min
125/300 | ETA 118.6 min
150/300 | ETA 100.6 min
175/300 | ETA 85.5 min
200/300 | ETA 68.6 min
225/300 | ETA 51.7 min
250/300 | ETA 34.4 min
275/300 | ETA 17.4 min
300/300 | ETA 0.0 min

=== Chain of Thought — metricas (soft-matching) ===
  EVALUACIÓN COMPLETA

── Recall & MRR (soft-matching) ──
Evaluable: 300 / 300
  Recall@1: 0.0133
  Recall@5: 0.0300
  MRR:      0.0206

── Novelty ──
  Novelty:  10.43 bits (norm: 0.668)
  No vistas en train: 18.5% (252/1365)

── BERTScore ──


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERTScore (275/300): P=0.8709  R=0.8821  F1=0.8764

{'Recall@1': 0.0133, 'Recall@5': 0.03, 'MRR': 0.0206, 'n_evaluable': 275, 'novelty_mean': 10.4285, 'novelty_norm': 0.6681, 'unseen_rate': 0.1846, 'n_dialogos': 275, 'n_recs': 1365, 'precision': 0.8709, 'recall': 0.8821, 'f1': 0.8764, 'n_total': 300}
